In [1]:
import sys
import os
import uuid

from pathlib import Path

SRC_PATH = Path.cwd().parents[2]

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(SRC_PATH)


/home/juanfgallo/Documents/projects/professional/magister/II semester/Dick data/final/src


In [2]:
import os
from dotenv import load_dotenv
from infrastructure.out.secrets_config.hf_token_config import HFTokenClass

load_dotenv()

HF_TOKEN = HFTokenClass().get_hf_token_from_call()

assert HF_TOKEN
print("HF token cargado desde .env")

HF token cargado desde .env


In [4]:
import uuid

RUN_ID = str(uuid.uuid4())

LIMIT_PER_SHARD = None

# Recomendado para la corrida completa: no tan agresivo.
TARGET_CONCURRENCY = 4
GLOBAL_CALLS_PER_MINUTE = 600
LOCAL_OUTPUT_PATH = str(Path("./hf_carddata_bronze_json").resolve())

print("RUN_ID:", RUN_ID)
print("LIMIT_PER_SHARD:", LIMIT_PER_SHARD)
print("TARGET_CONCURRENCY:", TARGET_CONCURRENCY)
print("GLOBAL_CALLS_PER_MINUTE:", GLOBAL_CALLS_PER_MINUTE)
print("LOCAL_OUTPUT_PATH:", LOCAL_OUTPUT_PATH)

RUN_ID: 89206b09-5849-473f-b1dd-d3a06ae43418
LIMIT_PER_SHARD: None
TARGET_CONCURRENCY: 4
GLOBAL_CALLS_PER_MINUTE: 600
LOCAL_OUTPUT_PATH: /home/juanfgallo/Documents/projects/professional/magister/II semester/Dick data/final/src/infrastructure/in/notebooks/hf_carddata_bronze_json


In [8]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("hf-carddata-co2-local")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.speculation", "false")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Timezone:", spark.conf.get("spark.sql.session.timeZone"))

spark

Spark version: 3.5.1
Timezone: UTC


In [9]:
import os
import sys
import json
import uuid
import logging
import warnings
from pathlib import Path
from itertools import islice
from datetime import datetime, timezone
from typing import Any, Iterator

from dotenv import load_dotenv
from huggingface_hub import HfApi

from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType,
    TimestampType,
)

/home/juanfgallo/Documents/projects/professional/magister/II semester/Dick data/final/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub.repocard").setLevel(logging.ERROR)

warnings.filterwarnings(
    "ignore",
    message="Invalid model-index. Not loading eval results into CardData."
)

In [11]:
RUN_ID = str(uuid.uuid4())

LOCAL_OUTPUT_PATH = str(Path("./hf_carddata_bronze_json").resolve())

TARGET_CONCURRENCY = 4
GLOBAL_CALLS_PER_MINUTE = 600

# Modo prueba:
LIMIT_PER_SHARD = 10

# Para corrida completa:
# LIMIT_PER_SHARD = None

print("RUN_ID:", RUN_ID)
print("LOCAL_OUTPUT_PATH:", LOCAL_OUTPUT_PATH)
print("TARGET_CONCURRENCY:", TARGET_CONCURRENCY)
print("GLOBAL_CALLS_PER_MINUTE:", GLOBAL_CALLS_PER_MINUTE)
print("LIMIT_PER_SHARD:", LIMIT_PER_SHARD)

RUN_ID: 6c37809e-1793-446f-b800-3317a8d47bd1
LOCAL_OUTPUT_PATH: /home/juanfgallo/Documents/projects/professional/magister/II semester/Dick data/final/src/infrastructure/in/notebooks/hf_carddata_bronze_json
TARGET_CONCURRENCY: 4
GLOBAL_CALLS_PER_MINUTE: 600
LIMIT_PER_SHARD: 10


In [12]:
EMISSION_SHARDS = [
    (0, 1),
    (1, 5),
    (5, 10),
    (10, 25),
    (25, 50),
    (50, 100),
    (100, 250),
    (250, 500),
    (500, 1_000),
    (1_000, 2_500),
    (2_500, 5_000),
    (5_000, 10_000),
    (10_000, 25_000),
    (25_000, 50_000),
    (50_000, 100_000),
    (100_000, 250_000),
    (250_000, 500_000),
    (500_000, 1_000_000),
    (1_000_000, 5_000_000),
    (5_000_000, None),
]

print("Número de shards:", len(EMISSION_SHARDS))

for i, (low, high) in enumerate(EMISSION_SHARDS):
    print(i, low, high)

Número de shards: 20
0 0 1
1 1 5
2 5 10
3 10 25
4 25 50
5 50 100
6 100 250
7 250 500
8 500 1000
9 1000 2500
10 2500 5000
11 5000 10000
12 10000 25000
13 25000 50000
14 50000 100000
15 100000 250000
16 250000 500000
17 500000 1000000
18 1000000 5000000
19 5000000 None


In [13]:
shards_df = spark.createDataFrame(
    [
        Row(
            shard_id=i,
            min_emissions=float(low),
            max_emissions=float(high) if high is not None else None,
        )
        for i, (low, high) in enumerate(EMISSION_SHARDS)
    ]
)

shards_df.printSchema()
shards_df.show(25, truncate=False)

print("Número de filas:", shards_df.count())
print("Particiones actuales:", shards_df.rdd.getNumPartitions())

root
 |-- shard_id: long (nullable = true)
 |-- min_emissions: double (nullable = true)
 |-- max_emissions: double (nullable = true)



+--------+-------------+-------------+
|shard_id|min_emissions|max_emissions|
+--------+-------------+-------------+
|0       |0.0          |1.0          |
|1       |1.0          |5.0          |
|2       |5.0          |10.0         |
|3       |10.0         |25.0         |
|4       |25.0         |50.0         |
|5       |50.0         |100.0        |
|6       |100.0        |250.0        |
|7       |250.0        |500.0        |
|8       |500.0        |1000.0       |
|9       |1000.0       |2500.0       |
|10      |2500.0       |5000.0       |
|11      |5000.0       |10000.0      |
|12      |10000.0      |25000.0      |
|13      |25000.0      |50000.0      |
|14      |50000.0      |100000.0     |
|15      |100000.0     |250000.0     |
|16      |250000.0     |500000.0     |
|17      |500000.0     |1000000.0    |
|18      |1000000.0    |5000000.0    |
|19      |5000000.0    |NULL         |
+--------+-------------+-------------+

Número de filas: 20
Particiones actuales: 8


In [14]:
schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("shard_id", IntegerType(), False),
    StructField("model_id", StringType(), True),

    StructField("emissions_threshold_min", DoubleType(), True),
    StructField("emissions_threshold_max", DoubleType(), True),

    StructField("card_data_json", StringType(), True),
    StructField("model_raw_json", StringType(), True),

    StructField("co2_emissions", StringType(), True),
    StructField("co2_source", StringType(), True),
    StructField("co2_training_type", StringType(), True),
    StructField("co2_geographical_location", StringType(), True),
    StructField("co2_hardware_used", StringType(), True),

    StructField("status", StringType(), False),
    StructField("error", StringType(), True),

    StructField("ingested_at", TimestampType(), False),
])

for field in schema.fields:
    print(field.name, field.dataType, "nullable=", field.nullable)

run_id StringType() nullable= False
shard_id IntegerType() nullable= False
model_id StringType() nullable= True
emissions_threshold_min DoubleType() nullable= True
emissions_threshold_max DoubleType() nullable= True
card_data_json StringType() nullable= True
model_raw_json StringType() nullable= True
co2_emissions StringType() nullable= True
co2_source StringType() nullable= True
co2_training_type StringType() nullable= True
co2_geographical_location StringType() nullable= True
co2_hardware_used StringType() nullable= True
status StringType() nullable= False
error StringType() nullable= True
ingested_at TimestampType() nullable= False


In [15]:
def to_json_safe(value: Any) -> str:
    def default_serializer(obj):
        if hasattr(obj, "to_dict"):
            return obj.to_dict()
        if hasattr(obj, "__dict__"):
            return obj.__dict__
        return str(obj)

    return json.dumps(
        value,
        default=default_serializer,
        ensure_ascii=False,
        sort_keys=True,
    )


def get_card_data_as_dict(card_data: Any) -> dict:
    if card_data is None:
        return {}

    if isinstance(card_data, dict):
        return card_data

    if hasattr(card_data, "to_dict"):
        return card_data.to_dict()

    if hasattr(card_data, "__dict__"):
        return card_data.__dict__

    return {}


def get_co2_values(card_data: dict) -> dict:
    co2 = card_data.get("co2_eq_emissions")

    if co2 is None:
        return {
            "emissions": None,
            "source": None,
            "training_type": None,
            "geographical_location": None,
            "hardware_used": None,
        }

    if isinstance(co2, (int, float)):
        return {
            "emissions": str(co2),
            "source": None,
            "training_type": None,
            "geographical_location": None,
            "hardware_used": None,
        }

    if isinstance(co2, dict):
        return {
            "emissions": str(co2.get("emissions")) if co2.get("emissions") is not None else None,
            "source": co2.get("source"),
            "training_type": co2.get("training_type"),
            "geographical_location": co2.get("geographical_location"),
            "hardware_used": co2.get("hardware_used"),
        }

    if hasattr(co2, "to_dict"):
        return get_co2_values({"co2_eq_emissions": co2.to_dict()})

    if hasattr(co2, "__dict__"):
        return get_co2_values({"co2_eq_emissions": co2.__dict__})

    return {
        "emissions": str(co2),
        "source": None,
        "training_type": None,
        "geographical_location": None,
        "hardware_used": None,
    }

In [16]:
test_cases = [
    {"co2_eq_emissions": {"emissions": 0.03967468113268738}},
    {"co2_eq_emissions": 0.9413042739759596},
    {"co2_eq_emissions": None},
    {},
]

for case in test_cases:
    print(case)
    print(get_co2_values(case))
    print("-----")

{'co2_eq_emissions': {'emissions': 0.03967468113268738}}
{'emissions': '0.03967468113268738', 'source': None, 'training_type': None, 'geographical_location': None, 'hardware_used': None}
-----
{'co2_eq_emissions': 0.9413042739759596}
{'emissions': '0.9413042739759596', 'source': None, 'training_type': None, 'geographical_location': None, 'hardware_used': None}
-----
{'co2_eq_emissions': None}
{'emissions': None, 'source': None, 'training_type': None, 'geographical_location': None, 'hardware_used': None}
-----
{}
{'emissions': None, 'source': None, 'training_type': None, 'geographical_location': None, 'hardware_used': None}
-----


In [21]:
def fetch_carddata_partition(rows: Iterator[Row]) -> Iterator[Row]:
    import time
    import logging
    import warnings

    logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
    logging.getLogger("huggingface_hub.repocard").setLevel(logging.ERROR)

    warnings.filterwarnings(
        "ignore",
        message="Invalid model-index. Not loading eval results into CardData."
    )

    hf_api = HfApi(token=HF_TOKEN)

    calls_per_minute_per_partition = GLOBAL_CALLS_PER_MINUTE / TARGET_CONCURRENCY
    sleep_seconds = 60.0 / calls_per_minute_per_partition

    for row in rows:
        shard_id = int(row["shard_id"])
        min_emissions = float(row["min_emissions"])
        max_emissions = row["max_emissions"]

        thresholds = (
            min_emissions,
            float(max_emissions) if max_emissions is not None else None,
        )

        try:
            models_iter = hf_api.list_models(
                emissions_thresholds=thresholds,
                cardData=True,
                full=True,
            )

            if LIMIT_PER_SHARD is not None:
                models_iter = islice(models_iter, int(LIMIT_PER_SHARD))

            produced = 0

            for model in models_iter:
                produced += 1

                card_data_dict = get_card_data_as_dict(model.cardData)
                co2 = get_co2_values(card_data_dict)

                yield Row(
                    run_id=RUN_ID,
                    shard_id=shard_id,
                    model_id=model.id,

                    emissions_threshold_min=min_emissions,
                    emissions_threshold_max=float(max_emissions) if max_emissions is not None else None,

                    card_data_json=to_json_safe(card_data_dict),
                    model_raw_json=to_json_safe(model),

                    co2_emissions=co2["emissions"],
                    co2_source=co2["source"],
                    co2_training_type=co2["training_type"],
                    co2_geographical_location=co2["geographical_location"],
                    co2_hardware_used=co2["hardware_used"],

                    status="ok",
                    error=None,

                    ingested_at=datetime.now(timezone.utc),
                )

                time.sleep(sleep_seconds)

            if produced == 0:
                yield Row(
                    run_id=RUN_ID,
                    shard_id=shard_id,
                    model_id=None,

                    emissions_threshold_min=min_emissions,
                    emissions_threshold_max=float(max_emissions) if max_emissions is not None else None,

                    card_data_json=None,
                    model_raw_json=None,

                    co2_emissions=None,
                    co2_source=None,
                    co2_training_type=None,
                    co2_geographical_location=None,
                    co2_hardware_used=None,

                    status="empty",
                    error=None,

                    ingested_at=datetime.now(timezone.utc),
                )

        except Exception as e:
            yield Row(
                run_id=RUN_ID,
                shard_id=shard_id,
                model_id=None,

                emissions_threshold_min=min_emissions,
                emissions_threshold_max=float(max_emissions) if max_emissions is not None else None,

                card_data_json=None,
                model_raw_json=None,

                co2_emissions=None,
                co2_source=None,
                co2_training_type=None,
                co2_geographical_location=None,
                co2_hardware_used=None,

                status="error",
                error=repr(e),

                ingested_at=datetime.now(timezone.utc),
            )

In [ ]:
test_rows = [
    Row(shard_id=0, min_emissions=0.0, max_emissions=1.0),
    Row(shard_id=1, min_emissions=1.0, max_emissions=5.0),
]

test_output = list(fetch_carddata_partition(iter(test_rows)))

print("Filas producidas:", len(test_output))

for row in test_output[:10]:
    print(row.shard_id, row.model_id, row.co2_emissions, row.status, row.error)

In [23]:
test_shards_df = shards_df.filter("shard_id in (0, 1)")

print("Filas test_shards_df:", test_shards_df.count())
test_shards_df.show(truncate=False)

test_result_rdd = (
    test_shards_df
    .repartition(2)
    .rdd
    .mapPartitions(fetch_carddata_partition)
)

debug_rows = test_result_rdd.take(10)

print("Filas desde RDD:", len(debug_rows))

for row in debug_rows:
    print(row.shard_id, row.model_id, row.co2_emissions, row.status, row.error)

Filas test_shards_df: 2
+--------+-------------+-------------+
|shard_id|min_emissions|max_emissions|
+--------+-------------+-------------+
|0       |0.0          |1.0          |
|1       |1.0          |5.0          |
+--------+-------------+-------------+



[Stage 25:>                                                         (0 + 1) / 1]

Filas desde RDD: 10
1 Anorak/nirvana 4.214012748213151 ok None
1 JushBJJ/autonlp-bp-29016523 3.273303707756322 ok None
1 Monsia/autonlp-tweets-classification-23044997 4.819872182577655 ok None
1 alperiox/autonlp-user-review-classification-536415182 1.268309634217171 ok None
1 am4nsolanki/autonlp-text-hateful-memes-36789092 1.4280361775467445 ok None
1 amansolanki/autonlp-Tweet-Sentiment-Extraction-20114061 3.651199395353127 ok None
1 astarostap/autonlp-antisemitism-2-21194454 2.0686690092905224 ok None
1 bitmorse/autonlp-ks-530615016 2.2247356264808964 ok None
1 world-wide/sent-sci-irrelevance 3.667033499762825 ok None
1 ds198799/autonlp-predict_ROI_1-29797722 2.7516207978192737 ok None


In [24]:
test_result_df = spark.createDataFrame(test_result_rdd, schema=schema)

print("Total records:", test_result_df.count())

test_result_df.select(
    "shard_id",
    "model_id",
    "co2_emissions",
    "co2_source",
    "status",
    "error",
).show(50, truncate=False)

Total records: 20


[Stage 34:>                                                         (0 + 1) / 1]

+--------+-------------------------------------------------------+-------------------+----------+------+-----+
|shard_id|model_id                                               |co2_emissions      |co2_source|status|error|
+--------+-------------------------------------------------------+-------------------+----------+------+-----+
|1       |Anorak/nirvana                                         |4.214012748213151  |NULL      |ok    |NULL |
|1       |JushBJJ/autonlp-bp-29016523                            |3.273303707756322  |NULL      |ok    |NULL |
|1       |Monsia/autonlp-tweets-classification-23044997          |4.819872182577655  |NULL      |ok    |NULL |
|1       |alperiox/autonlp-user-review-classification-536415182  |1.268309634217171  |NULL      |ok    |NULL |
|1       |am4nsolanki/autonlp-text-hateful-memes-36789092        |1.4280361775467445 |NULL      |ok    |NULL |
|1       |amansolanki/autonlp-Tweet-Sentiment-Extraction-20114061|3.651199395353127  |NULL      |ok    |NULL |
|

In [25]:
(
    test_result_df
    .write
    .mode("append")
    .format("json")
    .partitionBy("run_id", "shard_id")
    .save(LOCAL_OUTPUT_PATH)
)

print("Escritura de prueba terminada en:", LOCAL_OUTPUT_PATH)
print("RUN_ID:", RUN_ID)

[Stage 36:=============================>                            (1 + 1) / 2]

Escritura de prueba terminada en: /home/juanfgallo/Documents/projects/professional/magister/II semester/Dick data/final/src/infrastructure/in/notebooks/hf_carddata_bronze_json
RUN_ID: 6c37809e-1793-446f-b800-3317a8d47bd1


In [26]:
written_df = spark.read.schema(schema).json(LOCAL_OUTPUT_PATH)

current_run_df = written_df.filter(F.col("run_id") == RUN_ID)

print("Registros escritos para este RUN_ID:", current_run_df.count())

current_run_df.groupBy("status").count().show(truncate=False)

current_run_df.groupBy("shard_id").count().orderBy("shard_id").show(100, truncate=False)

current_run_df.select(
    "run_id",
    "shard_id",
    "model_id",
    "co2_emissions",
    "co2_source",
    "ingested_at",
).show(50, truncate=False)

Registros escritos para este RUN_ID: 20
+------+-----+
|status|count|
+------+-----+
|ok    |20   |
+------+-----+

+--------+-----+
|shard_id|count|
+--------+-----+
|0       |10   |
|1       |10   |
+--------+-----+

+------------------------------------+--------+-------------------------------------------------------+-------------------+----------+-----------------------+
|run_id                              |shard_id|model_id                                               |co2_emissions      |co2_source|ingested_at            |
+------------------------------------+--------+-------------------------------------------------------+-------------------+----------+-----------------------+
|6c37809e-1793-446f-b800-3317a8d47bd1|0       |KoalaAI/Text-Moderation                                |0.03967468113268738|NULL      |2026-05-06 01:38:42.697|
|6c37809e-1793-446f-b800-3317a8d47bd1|0       |Fauzan/autonlp-judulberita-32517788                    |0.9413042739759596 |NULL      |2026-05-06 

In [ ]:

'''
FULL PRUEBA
'''

In [33]:
RUN_ID = str(uuid.uuid4())

LIMIT_PER_SHARD = None
TARGET_CONCURRENCY = 2
GLOBAL_CALLS_PER_MINUTE = 240

BATCH_SIZE = 2

print("RUN_ID:", RUN_ID)

RUN_ID: 95d146ed-8d55-4a92-a5f1-7b96d87dff50


In [ ]:
completed_batches = []
failed_batches = []

all_shard_ids = list(range(len(EMISSION_SHARDS)))

for batch_start in range(0, len(all_shard_ids), BATCH_SIZE):
    batch_shards = all_shard_ids[batch_start:batch_start + BATCH_SIZE]

    print(f"\nProcesando batch de shards: {batch_shards}")

    try:
        batch_df = (
            shards_df
            .filter(F.col("shard_id").isin(batch_shards))
            .repartition(len(batch_shards))
        )

        batch_rdd = (
            batch_df
            .rdd
            .mapPartitions(fetch_carddata_partition)
        )

        batch_result_df = spark.createDataFrame(batch_rdd, schema=schema)

        (
            batch_result_df
            .write
            .mode("append")
            .format("json")
            .partitionBy("run_id", "shard_id")
            .save(LOCAL_OUTPUT_PATH)
        )

        print(f"Batch terminado: {batch_shards}")
        completed_batches.append(batch_shards)

    except Exception as e:
        print(f"Batch falló {batch_shards}: {repr(e)}")
        failed_batches.append((batch_shards, repr(e)))


Procesando batch de shards: [0, 1]


[Stage 52:>   (0 + 4) / 4][Stage 55:>   (0 + 1) / 1][Stage 58:>   (0 + 2) / 2]

In [ ]:
print("Shards completados:", completed_shards)
print("Shards fallidos:", failed_shards)

written_df = spark.read.schema(schema).json(LOCAL_OUTPUT_PATH)

current_run_df = written_df.filter(F.col("run_id") == RUN_ID)

print("Total registros RUN_ID:", current_run_df.count())

current_run_df.groupBy("status").count().show(truncate=False)

current_run_df.groupBy("shard_id").count().orderBy("shard_id").show(100, truncate=False)

In [ ]:
full_shards_df = shards_df.repartition(TARGET_CONCURRENCY)

full_result_rdd = (
    full_shards_df
    .rdd
    .mapPartitions(fetch_carddata_partition)
)

full_result_df = spark.createDataFrame(full_result_rdd, schema=schema)

In [29]:
sample_rows = full_result_df.take(10)

for row in sample_rows:
    print(row.shard_id, row.model_id, row.co2_emissions, row.status, row.error)

[Stage 50:>                                                         (0 + 1) / 1]

5 Ajay191191/autonlp-Test-530014983 55.10196329868386 ok None
5 Crasher222/kaggle-comp-test 60.744727079482495 ok None
5 MadhurJindalWorkMail/autonlp-Gibb-Detect-515314387 70.95647633212745 ok None
5 XYHY/autonlp-123-478412765 69.86520391863117 ok None
5 adelgasmi/autonlp-kpmg_nlp-18833547 64.58945483765274 ok None
5 adrianmoses/autonlp-auto-nlp-lyrics-classification-19333717 88.89388195672073 ok None
5 anegi/autonlp-dialogue-summariztion-583416409 72.26141764997115 ok None
5 medA/autonlp-FR_another_test-565016091 70.54639641012226 ok None
5 muhtasham/autonlp-Doctor_DE-24595544 92.87363201770962 ok None
5 rexxar96/autonlp-roberta-large-finetuned-467612250 73.72876780772296 ok None


In [ ]:
(
    full_result_df
    .write
    .mode("append")
    .format("json")
    .partitionBy("run_id", "shard_id")
    .save(LOCAL_OUTPUT_PATH)
)

print("Escritura completa terminada en:", LOCAL_OUTPUT_PATH)
print("RUN_ID:", RUN_ID)

In [ ]:
written_df = spark.read.schema(schema).json(LOCAL_OUTPUT_PATH)

current_run_df = written_df.filter(F.col("run_id") == RUN_ID)

print("Registros escritos para este RUN_ID:", current_run_df.count())

current_run_df.groupBy("status").count().show(truncate=False)

current_run_df.groupBy("shard_id").count().orderBy("shard_id").show(100, truncate=False)

In [ ]:
current_run_df.filter(F.col("status") != "ok").select(
    "shard_id",
    "status",
    "error",
).show(100, truncate=False)

In [ ]:
print("RUN_ID:", RUN_ID)
print("Completed batches:", completed_batches)
print("Failed batches:", failed_batches)

In [ ]:
written_df = spark.read.schema(schema).json(LOCAL_OUTPUT_PATH)

current_run_df = written_df.filter(F.col("run_id") == RUN_ID)

print("Total registros RUN_ID:", current_run_df.count())

In [ ]:
current_run_df.groupBy("status").count().show(truncate=False)

In [ ]:
current_run_df.groupBy("status").count().show(truncate=False)

In [ ]:
current_run_df.filter(F.col("status") != "ok").select(
    "shard_id",
    "model_id",
    "status",
    "error",
).show(100, truncate=False)

In [ ]:
current_run_df.filter(F.col("status") == "ok").select(
    "shard_id",
    "model_id",
    "co2_emissions",
    "co2_source",
    "co2_training_type",
    "co2_geographical_location",
    "co2_hardware_used",
).show(100, truncate=False)

In [ ]:
retry_failed_batches = failed_batches.copy()
failed_batches = []

for batch_shards, previous_error in retry_failed_batches:
    print(f"\nReintentando batch {batch_shards}. Error anterior: {previous_error}")

    try:
        batch_df = (
            shards_df
            .filter(F.col("shard_id").isin(batch_shards))
            .repartition(len(batch_shards))
        )

        batch_rdd = (
            batch_df
            .rdd
            .mapPartitions(fetch_carddata_partition)
        )

        batch_result_df = spark.createDataFrame(batch_rdd, schema=schema)

        (
            batch_result_df
            .write
            .mode("append")
            .format("json")
            .partitionBy("run_id", "shard_id")
            .save(LOCAL_OUTPUT_PATH)
        )

        print(f"Batch reintentado correctamente: {batch_shards}")
        completed_batches.append(batch_shards)

    except Exception as e:
        print(f"Batch volvió a fallar {batch_shards}: {repr(e)}")
        failed_batches.append((batch_shards, repr(e)))

In [ ]:
duplicates_df = (
    current_run_df
    .filter(F.col("status") == "ok")
    .groupBy("run_id", "shard_id", "model_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicados:", duplicates_df.count())
duplicates_df.show(100, truncate=False)